In [8]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os 

#FOLDER DEFINITION 
folder_path = "Medicycle_Datasets" 
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

#1. PATIENTS DATA 
patients_data = [
    {"patient_id": "P101", "full_name": "Kwame Owino", "phone_number": "+254710000001", "primary_address": "Westlands Estate", "home_region": "Nairobi", "caregiver_name": "Jane Wambui", "caregiver_phone": "+254790000001"},
    {"patient_id": "P102", "full_name": "Wanjiku Kamau", "phone_number": "+254710000002", "primary_address": "Kikuyu Estate", "home_region": "Kiambu", "caregiver_name": "Peter Omondi", "caregiver_phone": "+254790000002"},
    {"patient_id": "P103", "full_name": "Musa Otieno", "phone_number": "+254710000003", "primary_address": "Thika Road Estate", "home_region": "Nairobi", "caregiver_name": "Sarah Cherono", "caregiver_phone": "+254790000003"},
    {"patient_id": "P104", "full_name": "Achieng Anyango", "phone_number": "+254710000004", "primary_address": "South C Estate", "home_region": "Nairobi", "caregiver_name": "Hassan Ali", "caregiver_phone": "+254790000004"},
    {"patient_id": "P105", "full_name": "Fatuma Juma", "phone_number": "+254710000005", "primary_address": "Kilimani Estate", "home_region": "Nairobi", "caregiver_name": "Grace Nyambura", "caregiver_phone": "+254790000005"}
]
df_patients = pd.DataFrame(patients_data)

#2. PHARMACIES DATA
pharmacies_data = [
    {"pharmacy_id": "PH01", "pharmacy_name": "Halisi Pharma - Westlands", "region": "Nairobi", "stock_status": True},
    {"pharmacy_id": "PH02", "pharmacy_name": "Boma Chemist - Kikuyu", "region": "Kiambu", "stock_status": True},
    {"pharmacy_id": "PH03", "pharmacy_name": "Afya Bora - Thika Road", "region": "Nairobi", "stock_status": False}
]
df_pharmacies = pd.DataFrame(pharmacies_data)


#3. PRESCRIPTIONS DATA 
today = datetime.now()
prescriptions_data = [
    #Happy Path & Polypharmacy (Kwame Med 1)
    # 20 pills / 1 daily = 20 days left. Result: "STABLE"
    {"prescription_id": "RX001", "patient_id": "P101", "medication_name": "Amlodipine", 
     "daily_frequency": 1, "total_qty_dispensed": 30, "pills_remaining": 20, "is_verified": True, 
     "issue_date": (today - timedelta(days=10)).strftime('%Y-%m-%d'), "validity_months": 6, 
     "primary_pharmacy_id": "PH01", "document_url": "kwame_amlodipine.png"},
    
    #Stock Failover (Kwame Med 2)
    # 8 pills / 2 daily = 4 days left. Primary Pharmacy (PH03) is OUT OF STOCK. Result: "RE-ROUTED"
    {"prescription_id": "RX002", "patient_id": "P101", "medication_name": "Metformin", 
     "daily_frequency": 2, "total_qty_dispensed": 60, "pills_remaining": 8, "is_verified": True, 
     "issue_date": (today - timedelta(days=15)).strftime('%Y-%m-%d'), "validity_months": 6, 
     "primary_pharmacy_id": "PH03", "document_url": "kwame_metformin.png"},
    
    #5-Day Alert (Wanjiku)
    # 5 pills / 1 daily = 5 days left. Result: "REFILL ALERT"
    {"prescription_id": "RX003", "patient_id": "P102", "medication_name": "Atorvastatin", 
     "daily_frequency": 1, "total_qty_dispensed": 30, "pills_remaining": 5, "is_verified": True, 
     "issue_date": (today - timedelta(days=25)).strftime('%Y-%m-%d'), "validity_months": 3, 
     "primary_pharmacy_id": "PH02", "document_url": "wanjiku_atorvastatin.png"},
    
    #Safety Block/Expired (Musa)
    # Prescription is 210 days old (Validity 6 months = 180 days). Result: "EXPIRED / VISIT DOCTOR"
    {"prescription_id": "RX004", "patient_id": "P103", "medication_name": "Ventolin", 
     "daily_frequency": 1, "total_qty_dispensed": 30, "pills_remaining": 10, "is_verified": True, 
     "issue_date": (today - timedelta(days=210)).strftime('%Y-%m-%d'), "validity_months": 6, 
     "primary_pharmacy_id": "PH01", "document_url": "musa_ventolin.png"},
    
    #Verification Wall (Achieng)
    # is_verified is FALSE. Result: "PENDING UPLOAD"
    {"prescription_id": "RX005", "patient_id": "P104", "medication_name": "Insulin", 
     "daily_frequency": 2, "total_qty_dispensed": 60, "pills_remaining": 40, "is_verified": False, 
     "issue_date": (today - timedelta(days=10)).strftime('%Y-%m-%d'), "validity_months": 6, 
     "primary_pharmacy_id": "PH01", "document_url": ""},
    
    #Emergency (Fatuma)
    # 0 pills left. Result: "EMERGENCY / CAREGIVER ALERT"
    {"prescription_id": "RX006", "patient_id": "P105", "medication_name": "Losartan", 
     "daily_frequency": 1, "total_qty_dispensed": 30, "pills_remaining": 0, "is_verified": True, 
     "issue_date": (today - timedelta(days=30)).strftime('%Y-%m-%d'), "validity_months": 4, 
     "primary_pharmacy_id": "PH01", "document_url": "fatuma_losartan.png"}
]

df_rx = pd.DataFrame(prescriptions_data)

#DATA EXPORT
df_patients.to_csv(os.path.join(folder_path, "patients.csv"), index=False)
df_pharmacies.to_csv(os.path.join(folder_path, "pharmacies.csv"), index=False)
df_rx.to_csv(os.path.join(folder_path, "prescriptions.csv"), index=False)

print(f"Success! Data saved to {folder_path}")

Success! Data saved to Medicycle_Datasets


In [7]:
def run_medicycle_engine():
    # 1. LOAD DATASETS
    try:
        df_patients = pd.read_csv("patients.csv")
        df_rx = pd.read_csv("prescriptions.csv")
        df_pharma = pd.read_csv("pharmacies.csv")
    except FileNotFoundError:
        return "Error: Ensure patients.csv, prescriptions.csv, and pharmacies.csv are in this folder."

    today = datetime.now().date()
    execution_log = []

    # 2. PROCESS EACH TRACKED MEDICATION
    for _, rx in df_rx.iterrows():
        # Link to Patient and Primary Pharmacy
        patient = df_patients[df_patients['patient_id'] == rx['patient_id']].iloc[0]
        primary_pharma = df_pharma[df_pharma['pharmacy_id'] == rx['primary_pharmacy_id']].iloc[0]
        
        # LOGIC 1: VERIFICATION (Epic 2)
        if str(rx['is_verified']).upper() != "TRUE":
            status = "ACTION REQUIRED: PENDING VALIDATION"
            msg = f"Notify {patient['full_name']}: Please upload a clear photo of your prescription."
            trigger = "App Notification"
        
        else:
            # LOGIC 2: SAFETY & EXPIRY (Epic 2.3)
            issue_date = datetime.strptime(rx['issue_date'], '%Y-%m-%d').date()
            # Validity in months converted to days (approx 30 days/month)
            expiry_date = issue_date + pd.DateOffset(days=rx['validity_months'] * 30)
            
            if today > expiry_date.date():
                status = "SAFETY BLOCK: EXPIRED"
                msg = f"Alert {patient['full_name']}: Your script for {rx['medication_name']} is expired. Visit doctor."
                trigger = "System Lock"

            # LOGIC 3: DEPLETION & ROUTING (Epic 3 & 6)
            else:
                days_remaining = rx['pills_remaining'] / rx['daily_frequency']
                
                # Case: Emergency (0 Pills)
                if rx['pills_remaining'] <= 0:
                    status = "CRITICAL: DEPLETED"
                    msg = f"EMERGENCY SMS to Caregiver ({patient['caregiver_name']}): {patient['full_name']} is out of {rx['medication_name']}!"
                    trigger = "Caregiver SMS"
                
                # Case: Refill Window (5 Days Left)
                elif days_remaining <= 5:
                    # Check Pharmacy Stock
                    if primary_pharma['stock_status'] == True:
                        status = "REFILL TRIGGERED"
                        msg = f"Order for {rx['medication_name']} sent to {primary_pharma['pharmacy_name']}."
                        trigger = "Pharmacy API"
                    else:
                        # SMART ROUTING: Find another pharmacy in the same region
                        alt_pharma = df_pharma[(df_pharma['region'] == patient['home_region']) & 
                                              (df_pharma['stock_status'] == True)]
                        
                        if not alt_pharma.empty:
                            pharma_name = alt_pharma.iloc[0]['pharmacy_name']
                            status = "RE-ROUTED"
                            msg = f"Primary OOS. Order rerouted to {pharma_name} ({patient['home_region']})."
                            trigger = "Failover SMS"
                        else:
                            status = "OUT OF STOCK"
                            msg = f"No stock found in {patient['home_region']}. Support ticket raised."
                            trigger = "Support Alert"
                
                # Case: Stable
                else:
                    status = "STABLE"
                    msg = f"{int(days_remaining)} days of {rx['medication_name']} remaining."
                    trigger = "None"

        execution_log.append({
            "Patient": patient['full_name'],
            "Medication": rx['medication_name'],
            "System Status": status,
            "Engine Action": msg,
            "Channel": trigger
        })

    return pd.DataFrame(execution_log)

# RUN THE MVP DRY RUN
report = run_medicycle_engine()
print(report)

           Patient    Medication                        System Status  \
0      Kwame Owino    Amlodipine                               STABLE   
1      Kwame Owino     Metformin                            RE-ROUTED   
2    Wanjiku Kamau  Atorvastatin                     REFILL TRIGGERED   
3      Musa Otieno      Ventolin                SAFETY BLOCK: EXPIRED   
4  Achieng Anyango       Insulin  ACTION REQUIRED: PENDING VALIDATION   
5      Fatuma Juma      Losartan                   CRITICAL: DEPLETED   

                                       Engine Action           Channel  
0                   20 days of Amlodipine remaining.              None  
1  Primary OOS. Order rerouted to Halisi Pharma -...      Failover SMS  
2  Order for Atorvastatin sent to Boma Chemist - ...      Pharmacy API  
3  Alert Musa Otieno: Your script for Ventolin is...       System Lock  
4  Notify Achieng Anyango: Please upload a clear ...  App Notification  
5  EMERGENCY SMS to Caregiver (Grace Nyambura): F.